In [1]:
#Note to self: create function for dealing with paragraphs so that it is not repeated code

In [17]:
import requests
import re
from bs4 import BeautifulSoup, Tag
import pandas as pd
import pickle
#import html2text
#h = html2text.HTML2Text()

In [18]:
# Opening the html file
HTMLFile = open("Section 2.6 - MAT 2000_ Introduction to Statistics _ zyBooks.html", "r", encoding='utf-8')
  
# Reading the file
index = HTMLFile.read()
  
# Creating a BeautifulSoup object
S = BeautifulSoup(index)

HTMLFile.close()

In [19]:
#print(S.prettify)

In [20]:
tags = S.select('title, h3,' +
                'div.prompt-area,' +
                'p, table, ul, script' +
                'ol, img, div.activity-title,' +
                'div.static-container-title,' +
                'div.container-resource-payload')
#tags[6:]

In [21]:
def inside(tag):
    if tag.name == None:
        return ""
    for a in tag.find_all("a"):
        if a.has_attr('target'):
            del a['target']
        a.name = "url"
    for b in tag.find_all("b"):
        b.name = "term"
    for i in tag.find_all("i"):
        i.name = "em"
    for sp in tag.select("span.term"):
        sp.name = "term"
        sp.attrs = {}
    for m in tag.select("span.MathJax_Preview, span.MathJax_SVG"):
        m.decompose()
    for m in tag.select("script"):
        m.name = "m"
        m.attrs = {}
    for d in tag.select("div.table"):
        d.unwrap()
    for tabl in tag.select("table"):
        tag = tabl
        tabl.parent.unwrap()
    for tab in tag.select("tbody"):
        tab.name = "tabular"
    for row in tag.select("tr"):
        row.name = "row"
        if row.find("th"):
            row['header']='yes' 
        for c in row.select("th, td"):
            c.name = "cell"
    for im in tag.select("img"):
        im.name = "image"
        im['source'] = im['src']
        descr = S.new_tag(name = "description")
        descr.string = im['alt'] + "\n"
        im.append("\n")
        im.append(descr)
        im.append("\n")
        del im['alt'], im['src']
        if im.has_attr('class'):
            del im['class']
    if tag.name == "img":
        tag.name = "image"
        tag['source'] = tag['src']
        descr = S.new_tag(name = "description")
        descr.string = tag['alt'] + "\n"
        tag.append("\n")
        tag.append(descr)
        tag.append("\n")
        del tag['alt'], tag['src']
    return str(tag) + '\n'

In [22]:
t = tags[11]
#print(inside(t))
print(t)

<div class="container-resource-payload">
<p>
Do students pick which tire went flat in equal proportions? A common conjecture is that students tend to answer "front right" more often than would be expected by chance.
        </p><!-- --><!-- -->
</div>


In [23]:
for d in t.find_all('div'):
    d.unwrap()
    print(t)

In [24]:
def end_div(definition):
    if definition:
        definition = False
        return("</statement>\n</definition>\n\n")
    return("</remark>\n\n")

In [25]:
definition = False
new_sub = True

In [26]:
t = tags[11]
if t.name == 'div':
    for d in t.find_all('div'): #Extract inner divs
        d.unwrap()
    if t['class'] == ['activity-title']:
        s1 = re.sub("\s*\d+.\d+.\d+:\s*", "", t.text)
        s2 = re.sub(".\s*$", "", s1)
        s = ("<exercise>\n <title> " +
             s2 + "</title>\n")
    elif 'prompt-area' in t['class']:
        n = t.findAll(string=True, recursive=False)
        print(re.sub('\s','', ''.join(n)),)
        s = ("<statement>\n" +
         ''.join([inside(t) for t in t.children]) +
        "</statement>\n</exercise>\n\n")
    elif t['class'] == ['static-container-title']:
        if "Definition" in t.text:
            s = "<definition>\n <statement>"
            definition = True
        else:
            s1 = re.sub("\s+", " ", t.text)
            s2 = re.sub("\.", "", s1)
            s = ("<remark> \n <title>" +
                 s2 + "</title>\n")
    elif t['class'] == ['container-resource-payload']:
        n = t.findAll(string=True, recursive=False)
        print(re.sub('\s','', ''.join(n)),)
        s = (''.join([inside(t) for t in t.children]) +
             end_div(definition))
        definition = False
    else:
        s = re.sub("\s+", " ", t.text)
elif t.name == "h3":
    if new_sub:
        s = ("<subsection> \n <title>" +
             t.text +
             "</title>\n")
        new_sub = False
    else:
        s = ("</subsection>\n\n<subsection> \n <title>" +
             t.text +
             "</title>\n")
elif t.name in ['p', 'ol', 'ul', 'img', 'table']:
    if t.parent.name == "div" and 'html-content-resource' not in t.parent['class']:
        print('continue')
    elif t.parent.name == 'p':
        print('contiune')
    s = inside(t)
print(s)


<p>
Do students pick which tire went flat in equal proportions? A common conjecture is that students tend to answer "front right" more often than would be expected by chance.
        </p>
</remark>




In [11]:
#Title
title = tags[0].get_text().replace(' | zyBooks', '')

#Subtitle
subtitle = '<title>A worksheet for Grace College from Zybooks</title>\n'

In [12]:
#filename
#This is one for ordinary sections
#xml = re.sub(r'(S)ection (\d+)\.(\d+)(.+)', r'\1ection_\2_\3', title).lower()
#This is the one for the investigations
xml = re.sub(r'(S)ection (\d+)\.(\d+)(.+)', r'chapter_\2_investigation', title)
#xml = 'section_9_4'
filename = "../../source/" + xml + ".ptx"
print(filename)

../../source/chapter_2_investigation.ptx


In [13]:
#Turn list of tags into long string of html tags
pretext = ("<?xml version=\"1.0\" encoding=\"UTF-8\"?> \n\n<section xml:id=\"" +
            xml +
            "\" xmlns:xi=\"http://www.w3.org/2001/XInclude\">\n" +
            "<title>" + title + "</title>\n" +
            "<exercises>\n" +
            subtitle +
            "\n")
definition = False
new_sub = True
for t in tags[1:]:
    for d in t.find_all('div'): #Unwrap inner divs
        d.unwrap()
    if t.name == 'div':
        if t['class'] == ['activity-title']:
            s1 = re.sub("\s*\d*.\d*.\d*:\s*", "", t.text)
            s2 = re.sub(".\s*$", "", s1)
            s = ("<exercise>\n <title> " +
                 s2 + "</title>\n")
        elif 'prompt-area' in t['class']:
            n = t.findAll(string=True, recursive=False)
            print(re.sub('\s','', ''.join(n)),)
            s = ("<statement>\n" +
             ''.join([inside(t) for t in t.children]) +
            "</statement>\n</exercise>\n\n")
        elif t['class'] == ['static-container-title']:
            if "Definition" in t.text:
                s = "<definition>\n <statement>\n"
                definition = True
            else:
                s1 = re.sub("\s+", " ", t.text)
                s2 = re.sub("\.", "", s1)
                s = ("<remark> \n <title>" +
                     s2 + "</title>\n")
        elif t['class'] == ['container-resource-payload']:
            n = t.findAll(string=True, recursive=False)
            print(re.sub('\s','', ''.join(n)),)
            s = (''.join([inside(t) for t in t.children]) +
                 end_div(definition))
            definition = False
        else:
            s = re.sub("\s+", " ", t.text)
    elif t.name == "h3":
        if new_sub:
            s = ("<subsection> \n <title>" +
                 t.text +
                 "</title>\n")
            new_sub = False
        else:
            s = ("</subsection>\n\n<subsection> \n <title>" +
                 t.text +
                 "</title>\n")
    elif t.name == 'p':
        if t.parent.name == "div" and 'html-content-resource' not in t.parent['class']:
            continue
        s = inside(t)
    elif 'zyBooks' in t.attrs.values():
        continue #Do not include the zyBooks logo
    elif ['bread-crumbs'] in t.attrs.values():
        continue #Do not include the breadcrumbs
    elif ['level-1'] in t.attrs.values():
        continue #Do not include the level-1 sidebar
    elif ['level-2'] in t.attrs.values():
        continue #Do not include the level-2 sidebar
    elif t.name == 'ol' or t.name == 'ul':
        if t.parent.name == "div" and 'html-content-resource' not in t.parent['class']:
            continue
        elif t.parent.name == 'p':
            continue
        s = inside(t)
    pretext += re.sub("’", "'", s)
pretext += ("</subsection>\n\n</exercises>\n\n</section>")


Dostudentspickwhichtirewentflatinequalproportions?Acommonconjectureisthatstudentstendtoanswer"frontright"moreoftenthanwouldbeexpectedbychance.
Totesttheconjecture,aprofessorasked28studentstoimaginebeinginthesamesituationasthestudentsinthestoryandpickoneofthefourtires.Thedataissummarizedinthetablebelow.






The3SStrategywillhelpyouinvestigatehowmuchevidencethesampledataprovidesinsupportoftheresearchconjecture.










Anotherprofessorconductedasimilarstudywithasmallerclassandfoundthat7outof14studentspickedthefrontrighttire.


Supposeyoudidn’thaveapreconceivednotionthattherightfronttirewouldbechosenmoreoften,butinsteadjustwantedtofindoutifthefrontrighttirewaschosenataratethatwasdifferentthanone-fourth.Considertheoriginalsampleof28students.





In [14]:
ptx = open(filename, 'w', encoding='utf-8')
ptx.write(pretext)
ptx.close()

In [90]:
filename

'../../source/chapter_2_investigation.ptx'

In [14]:
print(pretext[:3000])

<?xml version="1.0" encoding="UTF-8"?> 

<section xml:id="chapter_2_investigation" xmlns:xi="http://www.w3.org/2001/XInclude">
<title>Section 2.6 - MAT 2000: Introduction to Statistics</title>
<exercises>
<title>A worksheet for Grace College from Zybooks</title>

<subsection> 
 <title>Tire study</title>
<remark> 
 <title> </title>
<p>A legendary story on college campuses concerns two students who missed an exam because of excessive partying, but they blamed the absence on a flat tire. The professor allowed the students to take a make-up exam in separate rooms. On the exam, the students were asked which tire was flat.</p>
</remark>

<remark> 
 <title> Step 1: Ask a research question </title>
</remark>

<remark> 
 <title> Step 2: Design a study and collect data </title>
<table>
<tabular><row header="yes">
<cell>Front left</cell>
<cell>Front right</cell>
<cell>Rear left</cell>
<cell>Rear right</cell>
</row>
<row>
<cell>6</cell>
<cell>14</cell>
<cell>4</cell>
<cell>4</cell>
</row>
</tabula

In [30]:
pretext.format??

Docstring:
S.format(*args, **kwargs) -> str

Return a formatted version of S, using substitutions from args and kwargs.
The substitutions are identified by braces ('{' and '}').
Type:      builtin_function_or_method